In [ ]:
# ===================================================================
# api.py - FastAPI Churn Prediction Service
# ===================================================================

# --- INSTALL MISSING PACKAGES (Uncomment if needed) ---
# import subprocess
# import sys
# 
# def install_package(package):
#     subprocess.check_call([sys.executable, "-m", "pip", "install", package])
# 
# try:
#     import fastapi
# except ImportError:
#     print("Installing FastAPI...")
#     install_package("fastapi")
# 
# try:
#     import uvicorn
# except ImportError:
#     print("Installing Uvicorn...")
#     install_package("uvicorn")

# --- IMPORTS ---
from fastapi import FastAPI, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional
import pandas as pd
import numpy as np
import pickle
import os
import json
from datetime import datetime

# ===================================================================
# CREATE FASTAPI APP
# ===================================================================

app = FastAPI(
    title="D2C Churn Prediction API",
    description="Predict customer churn for D2C personal-care brand",
    version="1.0.0"
)

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ===================================================================
# LOAD MODEL
# ===================================================================

model = None
scaler = None
feature_columns = None
model_name = None

def load_model():
    """Load the trained model from pickle file"""
    global model, scaler, feature_columns, model_name
    
    try:
        if os.path.exists('churn_model.pkl'):
            with open('churn_model.pkl', 'rb') as f:
                model_data = pickle.load(f)
            
            model = model_data.get('model')
            scaler = model_data.get('scaler')
            feature_columns = model_data.get('feature_columns')
            model_name = model_data.get('model_name', 'Unknown')
            
            print(f"✅ Model loaded: {model_name}")
            print(f"   Features: {len(feature_columns) if feature_columns else 0}")
            return True
        else:
            print("⚠️ Model file not found. Using dummy predictions.")
            return False
    except Exception as e:
        print(f"❌ Error loading model: {str(e)}")
        return False

# Load model on startup
load_model()

# ===================================================================
# PYDANTIC MODELS (Request/Response Schemas)
# ===================================================================

class ChurnRequest(BaseModel):
    """Single customer churn prediction request"""
    customer_id: str = Field(..., description="Unique customer identifier", example="CUST12345")
    features: Dict[str, Any] = Field(..., description="Customer feature dictionary")
    
    class Config:
        json_schema_extra = {
            "example": {
                "customer_id": "CUST12345",
                "features": {
                    "recency_days": 15,
                    "frequency_180d": 3,
                    "monetary_180d": 4500,
                    "sessions_30d": 12,
                    "last_visit_days_ago": 3,
                    "ticket_count_90d": 0
                }
            }
        }

class BatchChurnRequest(BaseModel):
    """Batch churn prediction request"""
    customers: List[ChurnRequest] = Field(..., description="List of customers to score")
    
    class Config:
        json_schema_extra = {
            "example": {
                "customers": [
                    {
                        "customer_id": "CUST_A",
                        "features": {
                            "recency_days": 5,
                            "frequency_180d": 8,
                            "monetary_180d": 8000
                        }
                    },
                    {
                        "customer_id": "CUST_B",
                        "features": {
                            "recency_days": 95,
                            "frequency_180d": 1,
                            "monetary_180d": 500
                        }
                    }
                ]
            }
        }

class ChurnResponse(BaseModel):
    """Single customer churn prediction response"""
    customer_id: str = Field(..., description="Customer identifier")
    churn_probability: float = Field(..., ge=0, le=1, description="Probability of churn (0-1)")
    will_churn: bool = Field(..., description="Will churn in next 60 days")
    risk_level: str = Field(..., description="Risk level: Low, Medium, High, Critical")
    recommended_action: str = Field(..., description="Recommended retention action")
    timestamp: str = Field(..., description="Prediction timestamp")
    
    class Config:
        json_schema_extra = {
            "example": {
                "customer_id": "CUST12345",
                "churn_probability": 0.34,
                "will_churn": False,
                "risk_level": "Low",
                "recommended_action": "No action needed - monitor normally",
                "timestamp": "2025-01-15T10:30:00.123456"
            }
        }

class BatchChurnResponse(BaseModel):
    """Batch churn prediction response"""
    predictions: List[ChurnResponse] = Field(..., description="List of predictions")
    summary: Dict[str, Any] = Field(..., description="Summary statistics")
    timestamp: str = Field(..., description="Response timestamp")

class HealthResponse(BaseModel):
    """Health check response"""
    status: str = Field(..., description="Service status")
    model_loaded: bool = Field(..., description="Whether model is loaded")
    model_name: str = Field(..., description="Model name")
    feature_count: int = Field(..., description="Number of features")
    timestamp: str = Field(..., description="Health check timestamp")

class FeaturesResponse(BaseModel):
    """Features list response"""
    feature_columns: List[str] = Field(..., description="Required feature columns")
    total_features: int = Field(..., description="Total number of features")
    model_loaded: bool = Field(..., description="Whether model is loaded")

# ===================================================================
# HELPER FUNCTIONS
# ===================================================================

def get_action_and_risk(probability: float) -> tuple:
    """
    Determine risk level and recommended action based on churn probability
    
    Args:
        probability: Churn probability (0-1)
    
    Returns:
        Tuple of (risk_level, recommended_action)
    """
    if probability < 0.3:
        return "Low", "No action needed - monitor normally"
    elif probability < 0.6:
        return "Medium", "Send engagement email with recommendations"
    elif probability < 0.8:
        return "High", "Send 15% discount code and proactive support"
    else:
        return "Critical", "Urgent: Send 25% discount + free shipping"

def prepare_features(features: Dict[str, Any]) -> pd.DataFrame:
    """
    Prepare features DataFrame for model prediction
    
    Args:
        features: Dictionary of feature values
    
    Returns:
        DataFrame with properly ordered features
    """
    if feature_columns is None:
        # Use provided features as-is
        return pd.DataFrame([features])
    
    # Create DataFrame
    df = pd.DataFrame([features])
    
    # Add missing columns with default values
    for col in feature_columns:
        if col not in df.columns:
            df[col] = 0
    
    # Select only required columns in correct order
    df = df[feature_columns]
    
    # Handle missing values
    for col in df.columns:
        if df[col].isna().any():
            df[col].fillna(0, inplace=True)
    
    return df

def make_prediction(features_df: pd.DataFrame) -> tuple:
    """
    Make prediction using loaded model
    
    Args:
        features_df: Prepared features DataFrame
    
    Returns:
        Tuple of (probability, prediction)
    """
    if model is None:
        # Dummy prediction based on recency
        recency = features_df['recency_days'].iloc[0] if 'recency_days' in features_df.columns else 30
        probability = min(0.9, recency / 100)
        prediction = 1 if probability >= 0.5 else 0
        return probability, prediction
    
    try:
        if scaler is not None:
            features_scaled = scaler.transform(features_df)
            probability = model.predict_proba(features_scaled)[0][1]
        else:
            probability = model.predict_proba(features_df)[0][1]
        
        prediction = 1 if probability >= 0.5 else 0
        return probability, prediction
    
    except Exception as e:
        print(f"Prediction error: {str(e)}")
        return 0.5, 0

# ===================================================================
# API ENDPOINTS
# ===================================================================

@app.get("/", response_model=Dict[str, Any])
async def root():
    """Root endpoint - API information"""
    return {
        "message": "D2C Churn Prediction API",
        "status": "operational" if model is not None else "degraded",
        "model": model_name or "not loaded",
        "version": "1.0.0",
        "endpoints": {
            "GET /": "API information",
            "GET /health": "Health check",
            "GET /features": "List required features",
            "POST /predict": "Single customer prediction",
            "POST /predict/batch": "Batch prediction for multiple customers"
        }
    }

@app.get("/health", response_model=HealthResponse)
async def health_check():
    """Health check endpoint"""
    return HealthResponse(
        status="healthy",
        model_loaded=model is not None,
        model_name=model_name or "not loaded",
        feature_count=len(feature_columns) if feature_columns else 0,
        timestamp=datetime.now().isoformat()
    )

@app.get("/features", response_model=FeaturesResponse)
async def get_features():
    """Get list of required features"""
    return FeaturesResponse(
        feature_columns=feature_columns or [],
        total_features=len(feature_columns) if feature_columns else 0,
        model_loaded=model is not None
    )

@app.post("/predict", response_model=ChurnResponse)
async def predict_churn(request: ChurnRequest):
    """
    Predict churn for a single customer
    
    Args:
        request: ChurnRequest with customer_id and features
    
    Returns:
        ChurnResponse with prediction results
    """
    try:
        # Prepare features
        features_df = prepare_features(request.features)
        
        # Make prediction
        probability, prediction = make_prediction(features_df)
        
        # Determine risk and action
        risk_level, action = get_action_and_risk(probability)
        
        return ChurnResponse(
            customer_id=request.customer_id,
            churn_probability=round(probability, 4),
            will_churn=bool(prediction),
            risk_level=risk_level,
            recommended_action=action,
            timestamp=datetime.now().isoformat()
        )
    
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=f"Invalid request: {str(e)}"
        )

@app.post("/predict/batch", response_model=BatchChurnResponse)
async def predict_churn_batch(request: BatchChurnRequest):
    """
    Predict churn for multiple customers
    
    Args:
        request: BatchChurnRequest with list of customers
    
    Returns:
        BatchChurnResponse with predictions and summary
    """
    try:
        predictions = []
        probabilities = []
        
        for customer in request.customers:
            # Prepare features
            features_df = prepare_features(customer.features)
            
            # Make prediction
            probability, prediction = make_prediction(features_df)
            risk_level, action = get_action_and_risk(probability)
            
            predictions.append(ChurnResponse(
                customer_id=customer.customer_id,
                churn_probability=round(probability, 4),
                will_churn=bool(prediction),
                risk_level=risk_level,
                recommended_action=action,
                timestamp=datetime.now().isoformat()
            ))
            probabilities.append(probability)
        
        # Calculate summary statistics
        summary = {
            "total_predictions": len(predictions),
            "avg_churn_probability": round(np.mean(probabilities), 4),
            "predicted_churn_count": sum(1 for p in predictions if p.will_churn),
            "risk_distribution": {
                "low": sum(1 for p in predictions if p.risk_level == "Low"),
                "medium": sum(1 for p in predictions if p.risk_level == "Medium"),
                "high": sum(1 for p in predictions if p.risk_level == "High"),
                "critical": sum(1 for p in predictions if p.risk_level == "Critical")
            }
        }
        
        return BatchChurnResponse(
            predictions=predictions,
            summary=summary,
            timestamp=datetime.now().isoformat()
        )
    
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=f"Invalid batch request: {str(e)}"
        )

# ===================================================================
# ERROR HANDLERS
# ===================================================================

@app.exception_handler(HTTPException)
async def http_exception_handler(request, exc):
    """Handle HTTP exceptions"""
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "error": exc.detail,
            "status_code": exc.status_code,
            "timestamp": datetime.now().isoformat()
        }
    )

@app.exception_handler(Exception)
async def general_exception_handler(request, exc):
    """Handle general exceptions"""
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={
            "error": "Internal server error",
            "detail": str(exc),
            "timestamp": datetime.now().isoformat()
        }
    )

# ===================================================================
# RUN THE APPLICATION
# ===================================================================

if __name__ == "__main__":
    import uvicorn
    
    print("="*60)
    print("🚀 Starting Churn Prediction API")
    print("="*60)
    print(f"Model loaded: {model is not None}")
    print(f"Model name: {model_name or 'dummy'}")
    print("="*60)
    print("📍 API will be available at: http://localhost:8000")
    print("📖 Documentation: http://localhost:8000/docs")
    print("="*60)
    
    uvicorn.run(app, host="0.0.0.0", port=8000)

✅ Model loaded: Logistic Regression
   Features: 25
🚀 Starting Churn Prediction API
Model loaded: True
Model name: Logistic Regression
📍 API will be available at: http://localhost:8000
📖 Documentation: http://localhost:8000/docs


/var/folders/4n/fgf1yppj15xg8_8hd7l4cqfw0000gn/T/ipykernel_3564/1806637756.py:98: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  customer_id: str = Field(..., description="Unique customer identifier", example="CUST12345")
/var/folders/4n/fgf1yppj15xg8_8hd7l4cqfw0000gn/T/ipykernel_3564/1806637756.py:96: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class ChurnRequest(BaseModel):
/var/folders/4n/fgf1yppj15xg8_8hd7l4cqfw0000gn/T/ipykernel_3564/1806637756.py:116: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprec

RuntimeError: asyncio.run() cannot be called from a running event loop